# Transportation-Distance Relaxation of Stochastic Dominance

The exact empirical stochastic-dominance model requires the portfolio return distribution to second-order stochastically dominate the benchmark distribution. In the preceding analysis, this condition was feasible for almost every historical estimation window, but the resulting dominance relation frequently failed to persist during the subsequent holding period.

This notebook implements the transportation-distance relaxation proposed by Dentcheva and Yi. Instead of introducing a separate slack variable for every stochastic-dominance condition, the method measures the distance between the portfolio return distribution and the complete set of distributions that second-order stochastically dominate the benchmark.

Let

$$
A_2(Y)
=
\left\{
Z:Z\succeq_{(2)}Y
\right\}
$$

denote the set of random variables that second-order stochastically dominate the benchmark \(Y\). For a portfolio return \(G(w)\), its distance to the dominating set is

$$
\operatorname{dist}
\left(
G(w),A_2(Y)
\right)
=
\inf_{Z\in A_2(Y)}
W_1\left(G(w),Z\right),
$$

where \(W_1\) is the first-order Wasserstein, or optimal mass transportation, distance.

For second-order stochastic dominance, this distance has the explicit representation

$$
D(w)
=
\max
\left\{
0,\;
\sup_{\eta\in\mathbb{R}}
\left[
\mathbb{E}
\left[
(\eta-G(w))_+
\right]
-
\mathbb{E}
\left[
(\eta-Y)_+
\right]
\right]
\right\}.
$$

The quantity \(D(w)\) has the following interpretation:

* \(D(w)=0\) if the portfolio already second-order stochastically dominates the benchmark;
* \(D(w)>0\) measures the minimum transportation distance required to move the portfolio distribution into the SSD-dominating set;
* the maximizing threshold \(\eta\) identifies the largest difference between the portfolio and benchmark integrated distribution functions.

The relaxed portfolio problem is

$$
\min_w
\left\{
-\mathbb{E}[G(w)]
+
\alpha D(w)
\right\},
$$

subject to

$$
\mathbf{1}^{\top}w=1,
\qquad
w\geq0.
$$

Equivalently, the model maximizes

$$
\mathbb{E}[G(w)]-\alpha D(w).
$$

The parameter \(\alpha\) controls the trade-off between expected return and proximity to the set of distributions that dominate SPY. A small value prioritizes estimated return, whereas a sufficiently large value places greater emphasis on eliminating the stochastic-dominance violation.


In [1]:
from pathlib import Path

import cvxpy as cp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

TRADING_DAYS = 252
ESTIMATION_YEARS = 3

BACKTEST_START_DATE = pd.Timestamp(
    "2010-04-12"
)

BENCHMARK_NAME = "SPY"
SOLVER = "CLARABEL"

DISTANCE_TOLERANCE = 1e-10

sns.set_theme(style="whitegrid")

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "etf_adjusted_close.csv"
)

prices = pd.read_csv(
    DATA_PATH,
    index_col="Date",
    parse_dates=["Date"],
).sort_index()

returns = (
    prices
    .pct_change(fill_method=None)
    .dropna()
)

assert BENCHMARK_NAME in returns.columns

# Continue excluding SPY from the decision
# universe for consistency with notebook 07.
asset_names = returns.columns.drop(
    BENCHMARK_NAME
)

backtest_start = returns.index[
    returns.index >= BACKTEST_START_DATE
][0]

backtest_end = returns.index.max()

print(
    "Return period:",
    returns.index.min(),
    "to",
    returns.index.max(),
)

print(
    "Backtest period:",
    backtest_start,
    "to",
    backtest_end,
)

print("Benchmark:", BENCHMARK_NAME)

print(
    "Investable assets:",
    asset_names.tolist(),
)

Return period: 2007-04-12 00:00:00 to 2020-04-01 00:00:00
Backtest period: 2010-04-12 00:00:00 to 2020-04-01 00:00:00
Benchmark: SPY
Investable assets: ['QQQ', 'IWM', 'EFA', 'EEM', 'IEF', 'TLT', 'LQD', 'HYG', 'GLD', 'DBC', 'VNQ']
